# 04 · Stage-1 generation (GPU scaffold)

**Execution status:** 🔴 GPU SCAFFOLD — UNEXECUTED (runs on Colab, RUNBOOK.md)
**Plan reference:** PLAN §III.9 (pilot) · §III.1 (factorial) · RUNBOOK sessions 1–3 · PREREG §10

The generation phase: a pilot **gate zero** (P0) and the full **4,630-unit factorial** (P1), both on `Qwen/Qwen2.5-Coder-7B-Instruct` via vLLM. **This notebook is a scaffold — no cell is executed here** (compute policy: no subject-model inference in this environment). Every model cell carries the `[GPU — COLAB ONLY]` banner with its runtime/VRAM, calls the tested `p19` APIs, and is left unrun. It runs on Colab session-by-session per `RUNBOOK.md`; the CPU-safe plan check already ran live in notebooks 00–01.

| | |
|---|---|
| **Inputs** | `config/*.yaml`, the frozen skill + prompts (both validated on CPU) |
| **Outputs** | 12 pilot gens (P0); 4,630 generation units + manifest (P1) |
| **Runtime** | P0 ≈ 0.5 L4-h · P1 ≈ 6 L4-h (see RUNBOOK session table) |

*Project 19 — Anatomy of a Design Skill. Governance: `CLAUDE.md`. Plan: `PLAN.md`. Derivations:
`THEORY.md`. Freeze: `PREREGISTRATION.md`. This notebook imports tested machinery from the `p19`
package and carries the narrative; it never re-implements logic that lives in `src/p19/`.*

> **Why unexecuted.** CLAUDE.md forbids loading Qwen weights, vLLM, or GPU code in this environment.
> The code below is complete, import-guarded, and unit-tested against a **plan builder** and mocks
> (`tests/test_manifest_generation.py`) — but left **unrun**. The per-notebook execution rule means
> *nothing* in this notebook is executed (not even the CPU-safe dry-run, which instead ran live in
> notebook 00). On Colab you run each banner cell top to bottom.

In [ ]:
%matplotlib inline
# Bootstrap: locate the repo root (repo-relative — no hardcoded paths) and make p19 importable.
import sys, pathlib
_here = pathlib.Path.cwd()
_root = next((c for c in [_here, *_here.parents]
              if (c / "pyproject.toml").exists() and (c / "src" / "p19").exists()), _here)
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from p19 import REPO_ROOT
np.random.seed(0)                      # notebook-level seed; every generator also takes an explicit seed
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 120)
print("p19 ready · repo:", REPO_ROOT.name)

## Session P0 · The pilot — gate zero (PLAN §III.9)

Before spending the generation budget we de-risk the core assumption ("the model can apply a five-part
skill"). **12 generations**: 2 prompts (L01 easy, D02 hard) × {FULL, NOSYS} × 3 seeds. **PASS iff**
render ≥ 11/12 AND FULL≠NOSYS on PSI + ≥2 objective families with a visible manual difference AND the
FULL−NOSYS POC gap > 0 on the majority of prompt×seed. **FAIL →** fix prompt formatting and re-pilot;
only if the skill still has no visible effect, trigger the sole sanctioned ADR-001 model fallback to
Llama-3.1-8B.

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# Expected: ~0.5 L4-hour · ~16 GB VRAM (bf16 7B) · 12 generations.
# Pilot generation via the vLLM runner (thin wrapper over p19.generation_vllm; NOT run here).
import os
from p19.generation_vllm import GenUnit, prompt_for_unit, run_generation
from p19.prompts import render_chatml

PILOT = [GenUnit(cell, pid, seed, "dev")
         for pid in ("L01", "D02") for cell in ("FULL", "NOSYS") for seed in (0, 1, 2)]
ARTIFACTS = os.environ["P19_ARTIFACTS"]                     # Google Drive (never the VM disk)
for u in PILOT:
    messages = prompt_for_unit(u)                           # ChatML: system=skill|none, user=brief+constraint
    prompt = render_chatml(messages)                        # tokenizer.apply_chat_template on Colab
    # vLLM .generate(prompt, SamplingParams(T=0.7, top_p=0.9, top_k=40, rep=1.05, seed=u.seed))
    # write {u.gen_id}.html to ARTIFACTS/pilot/  (see p19.generation_vllm.run_generation)
print("pilot: 12 units queued")

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# Expected: ~2 min CPU (render + metrics on the 12 pages) — runs on the GPU box or locally.
# Pilot acceptance check as code-ready ASSERTIONS (the numbers to eyeball; PREREG §10).
import concurrent.futures, pandas as pd
from p19.rendering import render_file
from p19 import poc

def _render(u):
    return render_file(f"{ARTIFACTS}/pilot/{u.gen_id}.html", out_dir=f"{ARTIFACTS}/pilot")
with concurrent.futures.ThreadPoolExecutor(4) as ex:
    R = list(ex.map(_render, PILOT))
rows = [dict(poc.assemble_row(r, r.screenshots["desktop"]), gen_id=u.gen_id,
             cell_id=u.cell_id, prompt_id=u.prompt_id, render_success=int(r.render_success))
        for u, r in zip(PILOT, R)]
M = poc.add_poc_column(pd.DataFrame(rows))

n_render = int(M["render_success"].sum())
full, nos = M[M.cell_id == "FULL"], M[M.cell_id == "NOSYS"]
psi_gap = nos["psi"].mean() - full["psi"].mean()            # FULL should have LOWER slop
poc_gap = full["poc"].mean() - nos["poc"].mean()
assert n_render >= 11, f"render {n_render}/12 < 11 -> fix prompt format, re-pilot"
assert psi_gap > 0,   "FULL not less-sloppy than NOSYS on PSI -> investigate before P1"
assert poc_gap > 0,   "FULL-NOSYS POC gap <= 0 -> skill has no visible effect (ADR-001 fallback trigger)"
print(f"PILOT PASS  render {n_render}/12 · PSI gap {psi_gap:+.3f} · POC gap {poc_gap:+.3f}")

## Session P1 · The full factorial — 4,630 units (PLAN §III.1)

Dev: all 24 cells × 40 prompts (headline/LOO/AOI @ 5 seeds, interaction @ 3) = **4,000**. Held-out:
{4 controls @5 + 5 LOO @3} × 18 prompts = **630**. Total **4,630**. Sampling is frozen (T=0.7,
top_p=0.9, top_k=40, rep-penalty=1.05, max_new_tokens=4096). The **plan builder is pure config and
CPU-safe** — its dry-run ran live in notebook 00; here it is shown (unrun) for continuity.

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# CPU-safe plan check (shown unrun to honor the per-notebook rule; it executed live in notebook 00).
from p19.generation_vllm import build_generation_plan, plan_summary, dry_run
plan = build_generation_plan("all")
summary = plan_summary(plan)                                 # {'dev':4000,'heldout':630,'total':4630,...}
assert summary["dev"] == 4000 and summary["heldout"] == 630 and summary["total"] == 4630
d = dry_run()                                                # + provenance: sampling hash, model, equalization
print(summary, "| model:", d["model"], "| sampling:", d["sampling_hash"][:12])

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# Expected: ~6 L4-hours · ~16 GB VRAM · ~2.2 GB artifacts to Drive. The Stage-1 bulk generation.
# Resume-safe (skips already-hashed units); checkpoints every 200 units (p19.generation_vllm).
from p19.generation_vllm import run_generation
ARTIFACTS, RESULTS = os.environ["P19_ARTIFACTS"], os.environ["P19_RESULTS"]
run_generation(out_dir=f"{ARTIFACTS}/stage1",
               manifest_path=f"{RESULTS}/stage1_manifest.jsonl",
               resume=True, split="dev")                     # then split="heldout" for the +630

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# P1 acceptance check (eyeball before P2). Cumulative 4,630; render-success >= 0.90 on a spot check.
from p19.manifest import Manifest
done = Manifest.load_jsonl(f"{RESULTS}/stage1_manifest.jsonl").hashed_paths()
assert len(done) == 4630, f"only {len(done)}/4630 generated -- resume run_generation to finish"
print(f"Stage-1 complete: {len(done)} units in the manifest")

## Manifest & resume logic (PLAN §VI.4/§VI.6)

Every unit's `gen_id` is a SHA-256 of `(cell, prompt, seed, split)`. `run_generation` reads the
manifest, **skips already-hashed units**, and resumes from the last Drive checkpoint — the ~90-minute
Colab idle disconnect is expected and handled by simply re-running the same command.

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# Resume demo -- on reconnect the runner skips finished units (idempotent; p19.manifest).
done = Manifest.load_jsonl(f"{RESULTS}/stage1_manifest.jsonl").hashed_paths()
remaining = [u for u in build_generation_plan("all")
             if f"{ARTIFACTS}/stage1/{u.gen_id}.html" not in done]
print(f"{len(remaining)} units remaining -> re-run run_generation() to continue from the checkpoint")

---
**Engine policy (ADR-002).** vLLM is used for Stage-1 bulk generation **only**; all Stage-2 activation
and steering work uses Hugging Face `.generate`. The two engines are **never cross-compared** — any
Stage-1↔Stage-2 numeric comparison re-generates the baseline in HF. Engine + version + sampling +
config hashes are stamped on every row (`manifest.run_manifest`, PREREG §11).

**Next:** notebook **05** (already executed) analyzes generations like these; the Stage-2 notebooks
(**06**, **07**) reuse the FULL/NEUTRAL dev generations as the extraction corpus. See `RUNBOOK.md`
sessions 1–3 for the exact Colab commands.